# Cognitive LLM — Phase 2: Kaggle A100 Ablation Study

This notebook runs the full ablation matrix on Kaggle A100:
- SmolLM3 3B in bf16 (no quantization)
- All block combinations from the ablation matrix
- Results logged to wandb
- Standardized benchmarks via lm-eval-harness

In [ ]:
!pip install -q torch transformers datasets accelerate bitsandbytes peft trl wandb lm-eval pyyaml

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import yaml

# Load config
with open('configs/smollm3_full.yaml') as f:
    config = yaml.safe_load(f)

model_id = config['model_id']
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Config loaded: {model_id}')

In [ ]:
from cognitive_llm.evaluation.ablation import ABLATION_MATRIX, AblationRunner
from cognitive_llm.models.cognitive_model import CognitiveModel

# Print the ablation matrix
print('Ablation Matrix:')
print(f'{"Experiment":<20} {"B1":>3} {"B2":>3} {"B3":>3} {"B4":>3} {"B5":>3} {"B6":>3}')
print('-' * 44)
for exp in ABLATION_MATRIX:
    print(
        f'{exp.name:<20} '
        f'{"ON" if exp.use_block1 else "-":>3} '
        f'{"ON" if exp.use_block2 else "-":>3} '
        f'{"ON" if exp.use_block3 else "-":>3} '
        f'{"ON" if exp.use_block4 else "-":>3} '
        f'{"ON" if exp.use_block5 else "-":>3} '
        f'{"ON" if exp.use_block6 else "-":>3}'
    )

In [ ]:
# Run each ablation experiment
from datasets import load_dataset
from torch.utils.data import DataLoader
from cognitive_llm.training.trainer import CognitiveTrainer
import wandb

# Load GSM8K
dataset = load_dataset('gsm8k', 'main', split='train')

def tokenize_gsm8k(examples):
    texts = [
        f'Question: {q}\nAnswer: {a}'
        for q, a in zip(examples['question'], examples['answer'])
    ]
    tokenized = tokenizer(
        texts, truncation=True, max_length=1024,
        padding='max_length', return_tensors='pt',
    )
    tokenized['labels'] = tokenized['input_ids'].clone()
    return tokenized

tokenized = dataset.map(tokenize_gsm8k, batched=True, remove_columns=dataset.column_names)
tokenized.set_format('torch')
train_loader = DataLoader(tokenized, batch_size=8, shuffle=True)

results = {}

for exp in ABLATION_MATRIX:
    print(f'\n{"="*60}')
    print(f'Running: {exp.name} (blocks: {exp.block_string()})')
    print(f'{"="*60}')
    
    # Load fresh model each time
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map='auto'
    )
    
    model = CognitiveModel(base_model, exp.to_config())
    
    trainer = CognitiveTrainer(
        model=model,
        train_dataloader=train_loader,
        config={
            **config.get('training', {}),
            'max_steps': 5000,
            'use_wandb': True,
        },
    )
    
    losses = trainer.train()
    results[exp.name] = {
        'blocks': exp.block_string(),
        'final_loss': losses[-1] if losses else None,
        'losses': losses,
    }
    
    # Free memory
    del model, base_model
    torch.cuda.empty_cache()

In [ ]:
# Compare results
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
for name, data in results.items():
    if data['losses']:
        ax.plot(data['losses'], label=f"{name} ({data['blocks']})")

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Ablation Study: Training Loss Comparison')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

# Print final losses
print(f'\n{"Experiment":<20} {"Blocks":<8} {"Final Loss":<12}')
print('-' * 40)
for name, data in results.items():
    loss_str = f"{data['final_loss']:.4f}" if data['final_loss'] else 'N/A'
    print(f"{name:<20} {data['blocks']:<8} {loss_str:<12}")

In [ ]:
# Run lm-eval-harness on best checkpoint
# Uncomment and adjust path to run:
# !lm_eval --model hf \
#     --model_args pretrained=./checkpoints/cognitive_smollm3 \
#     --tasks gsm8k,arc_challenge,hellaswag,mathqa \
#     --device cuda \
#     --batch_size 8 \
#     --output_path ./results/